### 准备数据集

In [5]:
# 加载tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('hfl/rbt3')

In [2]:
tokenizer

BertTokenizer(name_or_path='hfl/rbt3', vocab_size=21128, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [4]:
# 试用
tokenizer(['明月装饰了你的窗子', '你装饰了别人的梦'], 
                           truncation=True)

{'input_ids': [[101, 3209, 3299, 6163, 7652, 749, 872, 4638, 4970, 2094, 102], [101, 872, 6163, 7652, 749, 1166, 782, 4638, 3457, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}

In [6]:
# 从磁盘加载中文数据集
from datasets import load_from_disk

dataset = load_from_disk('./data/ChnSentiCorp/')

In [6]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 0
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1200
    })
})

In [7]:
# 缩小数据规模, 便于测试
dataset['train'] = dataset['train'].shuffle().select(range(2000))
dataset['test'] = dataset['test'].shuffle().select(range(100))

In [9]:
dataset['train'][0]

{'text': '对比了好长时间才选择的这款，配置还是不错的，做工很精致，外观全黑很漂亮，送的赠品也不错，摄像头很清晰，.奥特蓝星音质也很好，性价比高',
 'label': 1}

In [8]:
def f(data, tokenizer):
    return tokenizer(data['text'], truncation=True)

dataset = dataset.map(f, batched=True,
                     batch_size=1000,
                     num_proc=4,
                     remove_columns=['text'],
                     fn_kwargs={'tokenizer': tokenizer})

dataset

Map (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['label'],
        num_rows: 0
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 100
    })
})

In [9]:
# 删掉太长的句子
def f(data):
    return [len(i) <= 512 for i in data['input_ids']]

dataset = dataset.filter(f, batched=True, batch_size=1000, num_proc=4)

Filter (num_proc=4):   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

In [11]:
from transformers import AutoModelForSequenceClassification

In [12]:
model = AutoModelForSequenceClassification.from_pretrained('hfl/rbt3', num_labels=2)

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: hfl/rbt3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider traini

In [13]:
# 统计模型参数量
sum([i.nelement() for i in model.parameters()])

38478338

In [14]:
# 模型试算
import torch

# 模拟一条数据
data = {
    'input_ids': torch.ones(4, 10, dtype=torch.long),
    'token_type_ids': torch.ones(4, 10, dtype=torch.long),
    'attention_mask': torch.ones(4, 10, dtype=torch.long),
    'labels': torch.ones(4, dtype=torch.long)
}

out = model(**data)

In [16]:
out

SequenceClassifierOutput(loss=tensor(0.6901, grad_fn=<NllLossBackward0>), logits=tensor([[-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [17]:
out['loss']

tensor(0.6901, grad_fn=<NllLossBackward0>)

In [18]:
out['logits']

tensor([[-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885],
        [-0.0946, -0.0885]], grad_fn=<AddmmBackward0>)

In [15]:
# 定义评价函数
import evaluate

metric = evaluate.load('accuracy')

In [16]:
import numpy as np
from transformers.trainer_utils import EvalPrediction

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = logits.argmax(axis=1)
    # 不用metric, 自己算也行
    # return {'accuracy': (pred == labels).mean()}
    return metric.compute(predictions=pred, references=labels)

# 模拟输出
eval_pred = EvalPrediction(
    predictions=np.array([[0, 1], [2, 3], [4, 5], [6, 7]]),
    label_ids=np.array([1, 1, 0, 1])
)
compute_metrics(eval_pred)

{'accuracy': 0.75}

In [17]:
# 定义训练参数
from transformers import TrainingArguments

In [18]:
args = TrainingArguments(
    # 定义临时数据保存路径
    output_dir='./output_dir',

    # 定义测试执行策略, 可取 no, epoch, steps
    eval_strategy = 'steps',

    # 定义每隔多少个step执行一次测试
    eval_steps = 30,

    # 定义模型保存策略, 可取 no, epoch, steps
    save_strategy = 'steps',

    # 定义每隔多少个step保存一次
    save_steps = 30,

    # 定义总共训练集个轮次
    num_train_epochs = 1,

    # 定义学习率
    learning_rate = 1e-4,

    # 加入参数权重衰减, 防止过拟合
    weight_decay = 1e-2,

    # 定义训练和测试时候的批次大小
    per_device_eval_batch_size = 16,
    per_device_train_batch_size = 16,

    # 定义是否使用GPU训练
    use_cpu = False
 )

In [19]:
# 定义训练器
from transformers import Trainer
from transformers.data.data_collator import DataCollatorWithPadding

In [20]:
trainer = Trainer(
    model = model,
    args = args,
    train_dataset = dataset['train'],
    eval_dataset = dataset['test'],
    compute_metrics = compute_metrics,
    data_collator = DataCollatorWithPadding(tokenizer)
)

In [21]:
# 测试数据整理函数
data_collator = DataCollatorWithPadding(tokenizer)

In [23]:
# 获取一批数据
data = dataset['train'][:5]
print(data)

{'label': [0, 0, 1, 0, 1], 'input_ids': [[101, 1157, 4692, 2130, 8024, 6230, 2533, 2523, 671, 5663, 8024, 3766, 3300, 792, 5305, 784, 720, 2523, 3300, 817, 966, 4638, 5466, 1767, 6226, 1156, 511, 1372, 5543, 6432, 2523, 671, 5663, 4638, 741, 8024, 2130, 1059, 7479, 4143, 868, 1139, 3341, 4638, 511, 102], [101, 1355, 4178, 2661, 782, 8024, 2215, 1071, 3221, 1909, 1921, 4310, 1963, 671, 702, 4125, 3352, 8013, 1100, 1921, 6963, 1377, 809, 4689, 1343, 3265, 2797, 1901, 4638, 7178, 749, 511, 7946, 5682, 2218, 3221, 2900, 5292, 3119, 7415, 1690, 8024, 1506, 102], [101, 6821, 3221, 2769, 743, 4638, 5018, 671, 702, 3315, 3315, 8024, 1333, 3315, 2682, 2137, 691, 5698, 4638, 8024, 852, 3221, 3766, 3300, 6573, 8024, 6821, 3621, 2160, 2242, 4638, 2523, 4023, 778, 8024, 2802, 2458, 4692, 1168, 679, 3221, 5283, 7946, 4638, 8024, 3221, 3300, 4157, 3918, 4149, 5905, 4638, 8024, 2523, 1599, 3614, 511, 6163, 8766, 3683, 6772, 7937, 4172, 8024, 2462, 749, 671, 1288, 5905, 2242, 749, 8024, 738, 3766, 3300

In [24]:
# 查看句子长度
for i in data['input_ids']:
    print(len(i))

46
44
108
75
58


In [25]:
# 调用数据整理函数, 进行整理
data = data_collator(data)

In [26]:
# 查看整理之后的数据
for k, v in data.items():
    print(k, v.shape)

input_ids torch.Size([5, 108])
token_type_ids torch.Size([5, 108])
attention_mask torch.Size([5, 108])
labels torch.Size([5])


In [27]:
data

{'input_ids': tensor([[ 101, 1157, 4692, 2130, 8024, 6230, 2533, 2523,  671, 5663, 8024, 3766,
         3300,  792, 5305,  784,  720, 2523, 3300,  817,  966, 4638, 5466, 1767,
         6226, 1156,  511, 1372, 5543, 6432, 2523,  671, 5663, 4638,  741, 8024,
         2130, 1059, 7479, 4143,  868, 1139, 3341, 4638,  511,  102,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0],
        [ 101, 1355, 4178, 2661,  782, 8024, 2215, 1071, 3221, 1909, 1921, 4310,
         1963,  671,  702, 4125, 3352, 8013, 1100, 1921, 6963, 1377,  809, 4689,
         1343, 3265, 2797, 1901, 4638, 7178,  749,  511, 7946, 5682, 2218, 3221,
         2900

In [28]:
tokenizer.decode(data['input_ids'][0])

'[CLS] 刚 看 完 ， 觉 得 很 一 般 ， 没 有 介 绍 什 么 很 有 价 值 的 职 场 规 则 。 只 能 说 很 一 般 的 书 ， 完 全 靠 炒 作 出 来 的 。 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'

In [29]:
# 训练和测试

# 测试一把
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
No log,0.802837,0,0.510204


{'eval_loss': 0.8028373122215271, 'eval_accuracy': 0.5102040816326531}

In [30]:
# 训练
trainer.train()

Step,Training Loss,Validation Loss,Accuracy
30,No log,0.299044,0.897959
60,No log,0.231034,0.918367
90,No log,0.236659,0.897959
120,No log,0.187185,0.928571
124,No log,0.184881,0.928571


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=124, training_loss=0.44858289534045803, metrics={'train_runtime': 105.0895, 'train_samples_per_second': 18.851, 'train_steps_per_second': 1.18, 'total_flos': 71338102993548.0, 'train_loss': 0.44858289534045803, 'epoch': 1.0})

In [31]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
No log,0.184881,124,0.928571


{'eval_loss': 0.18488094210624695, 'eval_accuracy': 0.9285714285714286}

In [32]:
# 保存和加载模型
import torch

In [33]:
trainer.save_model(output_dir='./output_dir/save_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [40]:
# 加载 模型

# model.load_state_dict(torch.load('./output_dir/save_model/model.safetensors',map_location=lambda storage, loc: storage.cuda(1),weights_only=True))
from safetensors.torch import load_file

# 1. 使用专用的 load_file 函数
# 2. 注意：你的笔记本通常只有一块显卡，建议将 cuda(1) 改为 cuda(0)
state_dict = load_file('./output_dir/save_model/model.safetensors', device='cuda:0')

# 3. 加载到模型中
model.load_state_dict(state_dict)

<All keys matched successfully>

In [42]:
# 恢复中断训练
trainer.train(resume_from_checkpoint='./output_dir/checkpoint-90')

Step,Training Loss,Validation Loss,Accuracy
120,No log,0.187185,0.928571
124,No log,0.184881,0.928571


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=124, training_loss=0.10173170797286495, metrics={'train_runtime': 34.0415, 'train_samples_per_second': 58.194, 'train_steps_per_second': 3.643, 'total_flos': 71338102993548.0, 'train_loss': 0.10173170797286495, 'epoch': 1.0})

In [43]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Accuracy
No log,0.184881,124,0.928571


{'eval_loss': 0.18488097190856934, 'eval_accuracy': 0.9285714285714286}

In [45]:
# 预测
model.eval()

# 取一批数据做预测
for i, data in enumerate(trainer.get_eval_dataloader()):
    break

for k, v in data.items():
    # 因为model在gpu上训练的. 所以数据也要拷贝到gpu上
    data[k] = v.to('cuda')

# 预测
out = model(**data)
pred = out['logits'].argmax(dim=1)

In [46]:
pred

tensor([1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0], device='cuda:0')

In [49]:
for i in range(16):
    print(tokenizer.decode(data['input_ids'][i], skip_special_tokens=True))
    print('labels=', data['labels'][i].item())
    print('predict=', pred[i].item())

性 价 比 还 是 蛮 高 的 ， 此 本 本 在 本 地 卖 6999 ， 前 台 优 惠 99 元 ， 估 计 找 老 板 便 宜 个 300 ~ 500 就 差 不 多 了 ， 京 东 的 东 东 还 真 的 便 宜 。
labels= 1
predict= 1
啥 也 不 说 了 ， 就 是 便 宜 ， 昨 天 收 到 的 货 ， 质 量 还 不 确 定 ， 今 天 刚 装 了 xp 系 统 ， 这 样 才 像 电 脑 了 ， 用 起 来 很 方 便 。
labels= 1
predict= 0
1. 纯 黑 键 盘 ， 棕 色 外 面 板 ， 银 色 边 框 ， 耐 看 ， 做 工 也 不 错 ！ ~ 2. 奔 双 4200 ， 速 度 快 ， 显 卡 也 还 可 以 ， 能 应 付 主 流 游 戏 ， led 的 屏 幕 相 当 不 错 ， 显 示 效 果 好 ， 亮 度 高 ， 16 ： 9 的 新 宽 屏 幕 还 有 点 不 习 惯 ， 要 几 天 适 应 ！ 3. 发 货 速 度 很 快 ， 我 是 24 号 下 午 付 款 的 ， 收 款 确 认 后 不 到 2 个 小 时 就 发 货 完 毕 ， 效 率 高 ， 这 一 点 值 得 表 扬 ！ 4 ， 客 服 很 耐 心 ， 银 行 转 账 速 度 太 慢 ， 我 很 着 急 ， 他 耐 心 的 给 我 解 释 ！ 直 到 收 款 确 认 ， 谢 谢 那 位 同 志 。
labels= 1
predict= 1
闹 中 取 静, 虽 然 设 施 旧 了 一 点, 但 对 着 黄 浦 江 的 房 间 风 景 很 不 错, 唯 一 洗 澡 的 水 温 还 是 忽 冷 忽 热, 请 再 检 查 一 下, 这 个 问 题 比 较 严 重 的, 特 别 是 小 孩 洗 澡 的 时 候. 服 务 挺 好 的. 宾 馆 反 馈 2008 年 4 月 16 日 ： 首 先 感 谢 您 对 我 酒 店 风 景 和 服 务 的 肯 定, 酒 店 的 水 管 会 进 一 步 的 维 护 和 保 养, 完 善 现 有 硬 件 设 施, 谢 谢!
labels= 1
predict= 1
首 先 ， 交 通 相 对 来 说 ， 非 常 便 利 ， 从 quarry bay 地 铁 站 的 c 出 口 到 酒 店 不 超 